In [ ]:
import zipfile
import glob
from zipfile import BadZipFile


def unpack_zip(zip_path, extract_to):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

# Etap 1
# Rozpakowanie wszystkich plików zip znajdujących się w katalogu 'zip_files' do katalogu 'tmp'
for zip_file in glob.glob('zip_files/*.zip'):
    try:
        unpack_zip(zip_file, 'tmp/')
        print(f"Plik {zip_file} został rozpakowany")
    except BadZipFile:
        print(f"Plik {zip_file} jest uszkodzony")


Plik zip_files\2001_01_k.zip został rozpakowany
Plik zip_files\2001_02_k.zip został rozpakowany
Plik zip_files\2001_03_k.zip został rozpakowany
Plik zip_files\2001_04_k.zip został rozpakowany
Plik zip_files\2001_05_k.zip został rozpakowany
Plik zip_files\2001_06_k.zip został rozpakowany
Plik zip_files\2001_07_k.zip został rozpakowany
Plik zip_files\2001_08_k.zip został rozpakowany
Plik zip_files\2001_09_k.zip został rozpakowany
Plik zip_files\2001_10_k.zip został rozpakowany
Plik zip_files\2001_11_k.zip został rozpakowany
Plik zip_files\2001_12_k.zip został rozpakowany
Plik zip_files\2002_01_k.zip został rozpakowany
Plik zip_files\2002_02_k.zip został rozpakowany
Plik zip_files\2002_03_k.zip został rozpakowany
Plik zip_files\2002_04_k.zip został rozpakowany
Plik zip_files\2002_05_k.zip został rozpakowany
Plik zip_files\2002_06_k.zip został rozpakowany
Plik zip_files\2002_07_k.zip został rozpakowany
Plik zip_files\2002_08_k.zip został rozpakowany
Plik zip_files\2002_09_k.zip został rozp

In [ ]:
# Etap 2
# Usuwanie plików zawierających "t" w nazwie, czyli plików z danymi terminowymi
import os
for filename in os.listdir('tmp'):
    if 't' in filename:
        os.remove(os.path.join('tmp', filename))
        print(f"Plik {filename} został usunięty")


Plik k_d_t_01_2001.csv został usunięty
Plik k_d_t_01_2002.csv został usunięty
Plik k_d_t_01_2003.csv został usunięty
Plik k_d_t_01_2004.csv został usunięty
Plik k_d_t_01_2005.csv został usunięty
Plik k_d_t_01_2006.csv został usunięty
Plik k_d_t_01_2007.csv został usunięty
Plik k_d_t_01_2008.csv został usunięty
Plik k_d_t_01_2009.csv został usunięty
Plik k_d_t_01_2010.csv został usunięty
Plik k_d_t_01_2011.csv został usunięty
Plik k_d_t_01_2012.csv został usunięty
Plik k_d_t_01_2013.csv został usunięty
Plik k_d_t_01_2014.csv został usunięty
Plik k_d_t_01_2015.csv został usunięty
Plik k_d_t_01_2016.csv został usunięty
Plik k_d_t_01_2017.csv został usunięty
Plik k_d_t_01_2018.csv został usunięty
Plik k_d_t_01_2019.csv został usunięty
Plik k_d_t_01_2020.csv został usunięty
Plik k_d_t_01_2021.csv został usunięty
Plik k_d_t_01_2022.csv został usunięty
Plik k_d_t_01_2023.csv został usunięty
Plik k_d_t_01_2024.csv został usunięty
Plik k_d_t_02_2001.csv został usunięty
Plik k_d_t_02_2002.csv zo

In [ ]:
import glob
import os
import numpy as np
import pandas as pd

# ==========================================
# USTAWIENIA
# ==========================================
input_pattern = "tmp/*.csv"
output_file = "merged_data.csv"

columns = [
    "NSP", "POST", "ROK", "MC", "DZ",
    "TMAX", "WTMAX",
    "TMIN", "WTMIN",
    "STD", "WSTD",
    "TMNG", "WTMNG",
    "SMDB", "WSMDB",
    "ROOP",
    "PKSN", "WPKSN"
]

encodings_to_try = ["utf-8", "cp1250", "latin-1", "cp1252"]

# ==========================================
# FUNKCJA WCZYTAJACA JEDEN PLIK
# ==========================================
def read_single_csv(file_path, columns):
    if os.path.getsize(file_path) == 0:
        print(f"Plik pusty, pomijam: {file_path}")
        return None

    last_error = None

    for enc in encodings_to_try:
        try:
            df = pd.read_csv(
                file_path,
                sep=",",
                header=None,
                names=columns,
                encoding=enc,
                engine="python",
                skipinitialspace=True,
                na_values=["", " ", "NA", "N/A", "nan"],
                keep_default_na=True,
                on_bad_lines="skip"
            )

            # Jesli dataframe pusty po wczytaniu
            if df.empty:
                print(f"Plik po wczytaniu pusty, pomijam: {file_path}")
                return None

            # Czyszczenie spacji w kolumnach tekstowych
            for col in df.select_dtypes(include="object").columns:
                df[col] = (
                    df[col]
                    .astype(str)
                    .str.replace(r"\s+", " ", regex=True)
                    .str.strip()
                )

            # Zamiana pustych stringow na NaN
            df = df.replace(r"^\s*$", np.nan, regex=True)

            # Usuniecie wierszy calkowicie pustych
            df = df.dropna(how="all")

            # Jesli kolumna POST istnieje, wyczysc ja jeszcze raz
            if "POST" in df.columns:
                df["POST"] = (
                    df["POST"]
                    .astype(str)
                    .str.replace(r"\s+", " ", regex=True)
                    .str.strip()
                )

            # Jesli po czyszczeniu dataframe jest pusty
            if df.empty:
                print(f"Plik pusty po czyszczeniu, pomijam: {file_path}")
                return None

            print(f"Wczytano OK: {file_path} | encoding={enc} | rows={len(df)}")
            return df

        except Exception as e:
            last_error = e

    print(f"Nie udalo sie wczytac pliku: {file_path}")
    print(f"Ostatni blad: {last_error}")
    return None


# ==========================================
# GLOWNA CZESC
# ==========================================
all_files = glob.glob(input_pattern)

if not all_files:
    raise FileNotFoundError(f"Nie znaleziono plikow dla wzorca: {input_pattern}")

df_list = []

for file_path in all_files:
    df = read_single_csv(file_path, columns)
    if df is not None:
        df_list.append(df)

if not df_list:
    raise ValueError("Nie udalo sie wczytac zadnego pliku CSV.")

merged_df = pd.concat(df_list, ignore_index=True)

# ==========================================
# DODATKOWE CZYSZCZENIE
# ==========================================

# Usuniecie bialych znakow z nazw kolumn
merged_df.columns = [str(col).strip() for col in merged_df.columns]

# Oczyszczenie kolumny POST jeszcze raz na calej tabeli
if "POST" in merged_df.columns:
    merged_df["POST"] = (
        merged_df["POST"]
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

# Zamiana "nan" jako string na prawdziwe NaN
merged_df = merged_df.replace("nan", np.nan)

# Konwersja kolumn numerycznych
numeric_cols = [
    "NSP", "ROK", "MC", "DZ",
    "TMAX", "WTMAX",
    "TMIN", "WTMIN",
    "STD", "WSTD",
    "TMNG", "WTMNG",
    "SMDB", "WSMDB",
    "ROOP",
    "PKSN", "WPKSN"
]

for col in numeric_cols:
    if col in merged_df.columns:
        merged_df[col] = pd.to_numeric(merged_df[col], errors="coerce")

# Usuniecie rekordow bez podstawowych danych
required_cols = ["POST", "ROK", "MC", "DZ"]
merged_df = merged_df.dropna(subset=required_cols)

# Reset indeksu
merged_df = merged_df.reset_index(drop=True)

# ==========================================
# ZAPIS
# ==========================================
merged_df.to_csv(output_file, index=False, encoding="utf-8")

print("\n==========================================")
print("ZAKONCZONO")
print("==========================================")
print(f"Zapisano plik: {output_file}")
print(f"Liczba wierszy: {len(merged_df)}")
print(f"Liczba kolumn: {merged_df.shape[1]}")

# ==========================================
# KONTROLA JAKOSCI
# ==========================================
print("\nPodglad danych:")
print(merged_df.head(10))

print("\nInfo o danych:")
print(merged_df.info())

print("\nBraki danych:")
print(merged_df.isna().sum())

if "POST" in merged_df.columns:
    print("\nPrzykladowe nazwy stacji po czyszczeniu:")
    print(merged_df["POST"].dropna().drop_duplicates().sort_values().head(30))

Wczytano OK: tmp\k_d_01_2001.csv | encoding=cp1250 | rows=4929
Wczytano OK: tmp\k_d_01_2002.csv | encoding=cp1250 | rows=4898
Wczytano OK: tmp\k_d_01_2003.csv | encoding=cp1250 | rows=4805
Wczytano OK: tmp\k_d_01_2004.csv | encoding=cp1250 | rows=4925
Wczytano OK: tmp\k_d_01_2005.csv | encoding=cp1250 | rows=4898
Wczytano OK: tmp\k_d_01_2006.csv | encoding=cp1250 | rows=5022
Wczytano OK: tmp\k_d_01_2007.csv | encoding=cp1250 | rows=4991
Wczytano OK: tmp\k_d_01_2008.csv | encoding=cp1250 | rows=4991
Wczytano OK: tmp\k_d_01_2009.csv | encoding=cp1250 | rows=4991
Wczytano OK: tmp\k_d_01_2010.csv | encoding=cp1250 | rows=4867
Wczytano OK: tmp\k_d_01_2011.csv | encoding=cp1250 | rows=4898
Wczytano OK: tmp\k_d_01_2012.csv | encoding=cp1250 | rows=4929
Wczytano OK: tmp\k_d_01_2013.csv | encoding=cp1250 | rows=4867
Wczytano OK: tmp\k_d_01_2014.csv | encoding=cp1250 | rows=4867
Wczytano OK: tmp\k_d_01_2015.csv | encoding=cp1250 | rows=3627
Wczytano OK: tmp\k_d_01_2016.csv | encoding=cp1250 | ro

In [2]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1062629 entries, 0 to 1062628
Data columns (total 18 columns):
 #   Column  Non-Null Count    Dtype  
---  ------  --------------    -----  
 0   NSP     1062629 non-null  int64  
 1   POST    1062629 non-null  object 
 2   ROK     1062629 non-null  int64  
 3   MC      1062629 non-null  int64  
 4   DZ      1062629 non-null  int64  
 5   TMAX    1062593 non-null  float64
 6   WTMAX   505 non-null      float64
 7   TMIN    1062594 non-null  float64
 8   WTMIN   292 non-null      float64
 9   STD     1062592 non-null  float64
 10  WSTD    432 non-null      float64
 11  TMNG    1051557 non-null  float64
 12  WTMNG   390757 non-null   float64
 13  SMDB    1049065 non-null  float64
 14  WSMDB   527067 non-null   float64
 15  ROOP    525215 non-null   object 
 16  PKSN    1039587 non-null  float64
 17  WPKSN   872521 non-null   float64
dtypes: float64(12), int64(4), object(2)
memory usage: 145.9+ MB


In [3]:
merged_df.head()

,NSP,POST,ROK,MC,DZ,TMAX,WTMAX,TMIN,WTMIN,STD,WSTD,TMNG,WTMNG,SMDB,WSMDB,ROOP,PKSN,WPKSN
0,249180010,PSZCZYNA,2001,1,1,-1.3,NaN,-9.6,NaN,-5.7,NaN,-11.0,NaN,0.0,9.0,NaN,13.0,NaN
1,249180010,PSZCZYNA,2001,1,2,3.3,NaN,-12.0,NaN,-2.7,NaN,-13.2,NaN,0.0,9.0,NaN,11.0,NaN
2,249180010,PSZCZYNA,2001,1,3,1.5,NaN,-5.7,NaN,-1.5,NaN,-6.8,NaN,0.0,9.0,NaN,10.0,NaN
3,249180010,PSZCZYNA,2001,1,4,6.5,NaN,-1.1,NaN,1.9,NaN,-4.8,NaN,0.0,9.0,NaN,8.0,NaN
4,249180010,PSZCZYNA,2001,1,5,6.5,NaN,-2.3,NaN,2.8,NaN,-6.5,NaN,0.3,NaN,W,7.0,NaN
